# Classificação de Retinopatia Hipertensiva em Imagens de Fundo de Olho

## Resumo

A retinopatia hipertensiva é uma complicação ocular da hipertensão arterial sistêmica que, se não diagnosticada precocemente, pode evoluir para perda visual irreversível. Este trabalho apresenta um pipeline de aprendizado profundo para classificação binária automática (Normal vs. Retinopatia Hipertensiva) utilizando imagens de fundoscopia dos datasets **BRSET** (Nakayama et al., 2023) e **ODIR-5K**.

O problema apresenta três desafios técnicos centrais:

| Desafio | Estratégia adotada |
|---|---|
| Desbalanceamento severo (~18:1) | `WeightedRandomSampler` + Focal Loss (Lin et al., 2017) |
| Data leakage por paciente | Divisão estratificada por `patient_id` |
| Variabilidade de qualidade | Filtro `quality == "Adequate"` (metadado BRSET) |

| Atributo | Descrição |
|---|---|
| **Tarefa** | Classificação binária supervisionada |
| **Abordagem** | Transfer learning com ResNet-50 pré-treinado em ImageNet (He et al., 2016) |
| **Métrica principal** | AUC-ROC — robusta ao desbalanceamento de classes |
| **Plataforma** | Google Colab (GPU T4 / A100) |
| **Repositório** | https://github.com/Bappoz/fundus-classification |

---

## Estrutura do Documento

1. [Configuração do Ambiente](#1-configuracao)
2. [Verificação de Hardware](#2-hardware)
3. [Análise Exploratória dos Dados (EDA)](#3-eda)
4. [Divisão dos Dados](#4-split)
5. [Pipeline de Dados](#5-pipeline)
6. [Testes de Integração](#6-testes)

---

### Referências Principais

- Nakayama, L. F. et al. (2023). *BRSET: A Brazilian Multilabel Ophthalmological Dataset of Retina Fundus Photos*. PhysioNet.
- Lin, T.-Y. et al. (2017). *Focal Loss for Dense Object Detection*. ICCV 2017.
- He, K. et al. (2016). *Deep Residual Learning for Image Recognition*. CVPR 2016.
- Buslaev, A. et al. (2020). *Albumentations: Fast and Flexible Image Augmentations*. Information, 11(2), 125.
- Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning*. MIT Press. (cap. 11)
- Obermeyer, Z. & Emanuel, E. J. (2016). *Predicting the Future — Big Data, Machine Learning, and Clinical Medicine*. NEJM, 375(13).


---
## 1. Configuração do Ambiente

Detecta automaticamente o ambiente de execução (Colab vs. local), monta o Google Drive,
clona ou atualiza o repositório e instala as dependências.

Estrutura do pacote `src/retino/` no repositório:

    src/retino/
    ├── config.py        # configuração global (paths, hiperparâmetros)
    ├── losses.py        # FocalLoss (Lin et al., 2017)
    └── data/
        ├── loader.py    # build_labels(), filter_quality(), verify_files()
        ├── split.py     # split_by_patient(), split_report()
        ├── dataset.py   # FundusDataset, make_sampler(), get_loaders()
        └── transform.py # get_transforms() — Albumentations pipeline


### Datasets Utilizados

| Dataset | Fonte | Papel no estudo |
|---|---|---|
| **BRSET** | Brazilian Multilabel Ophthalmological Dataset (USP) | Normal (~8 457 imgs) + Hipertensiva (284 imgs) |
| **ODIR-5K** | Ocular Disease Intelligent Recognition Challenge | Hipertensiva adicional (193 imgs) |

#### Como obter o dataset

**Opção A — Colaborador do projeto:** solicite `dataset.zip` ao orientador e salve em `Meu Drive/`.

**Opção B — Download público:**
1. https://drive.google.com/file/d/1LL-8X1uCwgXRu0F_u2Q8mNDxEM2ylPcn/view?usp=sharing
2. Faça upload para `Meu Drive/` com o nome `dataset.zip`.

Estrutura esperada dentro do zip:

    dataset.zip
    └── retino/
        ├── meta/
        │   ├── labels_brset_filt.xlsx
        │   └── data_filt.xlsx
        └── images/
            ├── normal/
            ├── hr_brset/
            └── hr_odir5k/

> **Nome diferente?** Edite `DRIVE_ZIP` na célula abaixo.


In [ ]:
import os, sys, subprocess
from pathlib import Path

DRIVE_ZIP = "/content/drive/MyDrive/dataset.zip"
REPO      = "https://github.com/Bappoz/fundus-classification.git"

# Verifica integridade do numpy antes de qualquer import.
# Runtime com numpy parcialmente atualizado falha silenciosamente nos imports downstream.
try:
    import numpy._core.strings  # noqa: F401
except ImportError:
    print("=" * 60)
    print("ACAO NECESSARIA: numpy inconsistente.")
    print("  1. Runtime > Restart runtime")
    print("  2. Execute novamente a partir desta celula.")
    print("=" * 60)
    raise SystemExit("Reinicie o runtime antes de continuar.")

import numpy as _np
_NP_VERSION = _np.__version__

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Ambiente local — pulando mount do Drive.")

def _find_file(root, name, maxdepth=7):
    r = subprocess.run(
        ["find", root, "-maxdepth", str(maxdepth), "-name", name],
        capture_output=True, text=True
    )
    hits = [l for l in r.stdout.strip().splitlines() if l]
    return hits[0] if hits else None

if IN_COLAB:
    drive.mount("/content/drive")
    xlsx = _find_file("/content/drive/MyDrive", "labels_brset_filt.xlsx")
    if xlsx:
        print(f"Dataset ja extraido: {xlsx}")
    elif os.path.isfile(DRIVE_ZIP):
        print(f"Extraindo {DRIVE_ZIP}...")
        !unzip -q -o "{DRIVE_ZIP}" -d "/content/drive/MyDrive/"
        xlsx = _find_file("/content/drive/MyDrive", "labels_brset_filt.xlsx")
        if not xlsx:
            raise FileNotFoundError("labels_brset_filt.xlsx nao encontrado apos extracao.")
        print(f"Extracao concluida: {xlsx}")
    else:
        raise FileNotFoundError(f"Arquivo nao encontrado: {DRIVE_ZIP}")

    META_DIR   = Path(xlsx).parent
    IMAGES_DIR = META_DIR.parent / "images"
    if not IMAGES_DIR.is_dir():
        IMAGES_DIR = META_DIR
    print(f"  META_DIR   : {META_DIR}")
    print(f"  IMAGES_DIR : {IMAGES_DIR}")

    %cd /content
    if not os.path.isdir("/content/fundus-classification"):
        !git clone --quiet {REPO}
    else:
        print("Repositorio ja clonado — atualizando...")
        !cd /content/fundus-classification && git pull --quiet
    %cd /content/fundus-classification
    sys.path.insert(0, os.path.abspath("src"))
else:
    META_DIR = IMAGES_DIR = None
    src_path = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

subprocess.run(
    ["pip", "install", "-q", "timm", "albumentations", "openpyxl", f"numpy=={_NP_VERSION}"],
    check=True
)
print("Ambiente configurado com sucesso.")


---
## 2. Verificação de Hardware

Confirma disponibilidade de GPU e exibe a configuração global do experimento.
A configuração centralizada fica em
[`src/retino/config.py`](https://github.com/Bappoz/fundus-classification/blob/master/src/retino/config.py) — classe `Config`:

| Parâmetro | Valor | Justificativa |
|---|---|---|
| `img_size` | 224 | Padrão ImageNet — compatível com todos os backbones pré-treinados |
| `imagenet_mean/std` | (0.485, 0.456, 0.406) / (0.229, 0.224, 0.225) | Normalização ImageNet para fine-tuning estável |
| `batch_size` | 32 | Equilíbrio memória de GPU vs. estimativa do gradiente |
| `backbone` | `resnet50` | Boa relação capacidade/custo; amplamente validado em imagens médicas |

> Sem GPU: *Runtime → Change runtime type → GPU (T4)* antes de prosseguir.


In [ ]:
import torch
from retino.config import cfg

gpu_ok      = torch.cuda.is_available()
device_name = torch.cuda.get_device_name(0) if gpu_ok else "N/A"

print(f"GPU disponivel : {'Sim' if gpu_ok else 'Nao'}")
print(f"  Dispositivo  : {device_name}")
print(f"  Device cfg   : {cfg.device}")
print(f"  Backbone     : {cfg.backbone}")
print(f"  Img size     : {cfg.img_size}px")
print(f"  Batch size   : {cfg.batch_size}")
print(f"  Seed         : {cfg.seed}")

if not gpu_ok:
    raise RuntimeError("GPU nao disponivel — altere o runtime antes de prosseguir.")


---
## 3. Análise Exploratória dos Dados (EDA)

Objetivos:
1. **Quantificar o desbalanceamento** para dimensionar as estratégias de mitigação
2. **Identificar imagens de baixa qualidade** que possam introduzir ruído
3. **Inspecionar visualmente** amostras para verificar a diversidade clínica

### 3.1 Inventário do Dataset

Módulo: [`src/retino/data/loader.py`](https://github.com/Bappoz/fundus-classification/blob/master/src/retino/data/loader.py)

- `build_labels()` — unifica BRSET + ODIR-5K em schema `[path, patient_id, label, source, eye, quality]`
- `verify_files()` — descarta registros sem arquivo em disco
- `filter_quality()` — mantém `quality == "Adequate"` no BRSET; ODIR-5K mantido integralmente


In [ ]:
from retino.data.loader import build_labels, filter_quality, verify_files
from retino.config import cfg

if META_DIR is not None:
    cfg.meta_dir  = META_DIR
    cfg.image_dir = IMAGES_DIR

df = build_labels()
df = verify_files(df)
df = filter_quality(df)

n_total  = len(df)
n_normal = (df.label == 0).sum()
n_hiper  = (df.label == 1).sum()
ratio    = n_normal / n_hiper

print(f"Total de imagens  : {n_total:,}")
print(f"  Normal          : {n_normal:,}  ({100 * n_normal / n_total:.1f}%)")
print(f"  Hipertensiva    : {n_hiper:,}  ({100 * n_hiper / n_total:.1f}%)")
print(f"  Ratio neg:pos   : {ratio:.1f}:1")
print()
print("Distribuicao por fonte:")
src_table = (
    df.groupby(["source", "label"])
      .size()
      .rename("n")
      .rename({0: "normal", 1: "hipertensiva"}, level="label")
)
print(src_table.to_string())


### 3.2 Estrutura por Paciente e Controle de Data Leakage

Um problema clássico em datasets médicos é o **data leakage por paciente** (Obermeyer & Emanuel, 2016):
o mesmo paciente pode ter múltiplas imagens (olho esquerdo/direito, exames diferentes).
Se o split for feito por imagem aleatoriamente, imagens do mesmo paciente aparecem
simultaneamente no treino e no teste — o modelo aprende a reconhecer **o paciente**, não a
patologia, resultando em métricas infladas que não generalizam clinicamente.

A solução está na **Seção 4**.


In [ ]:
imgs_por_paciente = df.groupby("patient_id").size()

print(f"Pacientes unicos       : {df.patient_id.nunique():,}")
print(f"Imagens / paciente     : media={imgs_por_paciente.mean():.2f}  max={imgs_por_paciente.max()}")
print(f"Pacientes com > 1 img  : {(imgs_por_paciente > 1).sum():,}")
print()
print("Split DEVE ser por patient_id para evitar data leakage.")


### 3.2.1 Avaliação de Qualidade das Imagens (BRSET)

O BRSET fornece metadados de qualidade por imagem: `quality` (global), `focus` (nitidez),
`illum` (iluminação) e `artifacts`.

**Critério adotado:** apenas `quality == "Adequate"` é mantido.
Implementado em `filter_quality()` — [`src/retino/data/loader.py`](https://github.com/Bappoz/fundus-classification/blob/master/src/retino/data/loader.py).

Imagens do ODIR-5K não possuem metadados de qualidade e são mantidas integralmente.


In [ ]:
brset_df = df[df.source == "brset"]
print("=== quality ===")
print(brset_df["quality"].value_counts())
print("\n=== focus ===")
print(brset_df["focus"].value_counts())
print("\n=== illum ===")
print(brset_df["illum"].value_counts())
print("\n=== qualidade por label ===")
print(brset_df.groupby(["label", "quality"]).size())


### 3.3 Distribuição de Classes e Fontes

Com desbalanceamento ~18:1, um classificador ingênuo que preveja sempre "normal"
atingiria acurácia > 94% — métrica enganosa. Por isso utilizamos **AUC-ROC** e
**AUC-PR** como métricas primárias, invariantes à distribuição de classes.

**Estratégias de mitigação (detalhadas na Seção 5):**

1. **WeightedRandomSampler** — sorteio com probabilidade `1/freq(classe)`, batches ~1:1
   sem duplicar dados físicos. Ref: `make_sampler()` —
   [`src/retino/data/dataset.py`](https://github.com/Bappoz/fundus-classification/blob/master/src/retino/data/dataset.py)

2. **Focal Loss** (Lin et al., 2017) — fator `(1-pt)^gamma` reduz peso de exemplos fáceis;
   `alpha=0.75` pondera a classe positiva rara; `gamma=2.0` valor canônico dos autores.
   Ref: [`src/retino/losses.py`](https://github.com/Bappoz/fundus-classification/blob/master/src/retino/losses.py)

3. **Augmentação de dados** — exclusiva ao treino. Ref: `get_transforms('train')` —
   [`src/retino/data/transform.py`](https://github.com/Bappoz/fundus-classification/blob/master/src/retino/data/transform.py)


In [ ]:
import matplotlib.pyplot as plt

label_names = {0: "Normal", 1: "Hipertensiva"}
palette     = {"Normal": "#4C9BE8", "Hipertensiva": "#E85C4C"}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle("Distribuicao do Dataset", fontsize=14, fontweight="bold")

counts = df["label"].map(label_names).value_counts()
bars = axes[0].bar(
    counts.index, counts.values,
    color=[palette[k] for k in counts.index],
    edgecolor="white", linewidth=0.8, width=0.5
)
for bar, val in zip(bars, counts.values):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + max(counts.values) * 0.02,
        f"{val:,}", ha="center", va="bottom", fontsize=11, fontweight="bold"
    )
axes[0].set_title("Contagem por Classe", pad=10)
axes[0].set_ylabel("Num. de Imagens")
axes[0].set_ylim(0, max(counts.values) * 1.15)
axes[0].spines[["top", "right"]].set_visible(False)

src_pivot = (
    df.assign(classe=df["label"].map(label_names))
      .groupby(["source", "classe"])
      .size()
      .unstack(fill_value=0)
)
src_pivot.plot(
    kind="bar", ax=axes[1],
    color=[palette[c] for c in src_pivot.columns],
    edgecolor="white", linewidth=0.8, rot=30
)
axes[1].set_title("Contagem por Fonte e Classe", pad=10)
axes[1].set_xlabel("Fonte", labelpad=8)
axes[1].set_ylabel("Num. de Imagens")
axes[1].spines[["top", "right"]].set_visible(False)
axes[1].legend(title="Classe", framealpha=0.8)

plt.tight_layout()
plt.savefig("eda_distribuicao.png", dpi=150, bbox_inches="tight")
plt.show()


### 3.4 Amostras Visuais

Inspeção qualitativa de cada classe. Na retinopatia hipertensiva os achados clínicos
típicos incluem estreitamento arteriolar, cruzamentos arteriovenosos patológicos (sinal
de Gunn), hemorragias em chama de vela e exsudatos duros.

Pipeline de augmentação em `get_transforms('train')` —
[`src/retino/data/transform.py`](https://github.com/Bappoz/fundus-classification/blob/master/src/retino/data/transform.py)
via Albumentations (Buslaev et al., 2020):

| Transformação | Justificativa clínica |
|---|---|
| `HorizontalFlip` | Olho esquerdo e direito são espelhados |
| `VerticalFlip` | Diversidade de orientação de vasos |
| `Rotate(+-15 graus)` | Variação de angulação de captura |
| `RandomBrightnessContrast` | Variação de equipamentos e iluminação |
| `HueSaturationValue` | Diferenças de calibração entre câmeras |
| `GaussianBlur / GaussNoise` | Imperfeições ópticas e variação de sensor |

Validação e teste recebem apenas `Resize + Normalize` (condições realistas).


In [ ]:
import cv2
import matplotlib.pyplot as plt

def show_samples(df, n=4, seed=42):
    label_names = {0: "Normal", 1: "Hipertensiva"}
    row_colors  = {0: "#4C9BE8", 1: "#E85C4C"}
    fig, axes = plt.subplots(2, n, figsize=(3 * n, 7))
    fig.suptitle("Amostras: Normal vs Retinopatia Hipertensiva", fontsize=13, fontweight="bold")
    for row, lab in enumerate([0, 1]):
        subset = df[df.label == lab].sample(min(n, (df.label == lab).sum()), random_state=seed)
        for col, (_, r) in enumerate(subset.iterrows()):
            img = cv2.cvtColor(cv2.imread(str(r.path)), cv2.COLOR_BGR2RGB)
            axes[row, col].imshow(img)
            axes[row, col].axis("off")
            axes[row, col].set_title(r.source, fontsize=8, color="#555555")
        axes[row, 0].set_ylabel(
            label_names[lab], fontsize=12, fontweight="bold",
            color=row_colors[lab], rotation=90, labelpad=12
        )
    plt.tight_layout()
    plt.savefig("eda_amostras.png", dpi=150, bbox_inches="tight")
    plt.show()

show_samples(df)


---
## 4. Divisão dos Dados

### 4.1 Justificativa para Split por Paciente

Conforme a Seção 3.2, a divisão deve operar no nível do **paciente**, não da imagem.

A função `split_by_patient()` em
[`src/retino/data/split.py`](https://github.com/Bappoz/fundus-classification/blob/master/src/retino/data/split.py)
implementa:

1. **Zero leakage garantido** — `patient_id` aparece em exatamente um dos três splits
   (filtrado com `.isin()` após dividir o conjunto de IDs).

2. **Estratificação por rótulo de paciente** — label do paciente = `max(labels das imagens)`:
   paciente positivo = tem alguma imagem hipertensiva. Usa `stratify=p_labels` no
   `train_test_split` do scikit-learn para preservar a proporção nos três splits.

3. **Prefixos por fonte** — `brset_` e `odir_` no `patient_id` previnem colisões numéricas
   entre datasets.

### 4.2 Proporções Adotadas: 70 / 15 / 15

A divisão 70/15/15 é padrão consolidado para conjuntos de médio porte (Goodfellow et al., 2016):

| Split | Proporção | Papel |
|---|---|---|
| **Treino** | 70% | Maximiza dados para aprendizado do backbone |
| **Validação** | 15% | Ajuste de hiperparâmetros e early stopping |
| **Teste** | 15% | Estimativa não enviesada da performance |

A divisão ocorre em dois passos (dois `train_test_split` encadeados) para manter
a estratificação correta nos três splits simultaneamente.


In [ ]:
from retino.data.split import split_by_patient, split_report

train_df, val_df, test_df = split_by_patient(
    df,
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
    seed=cfg.seed,
)

print("=" * 58)
print(f"{'Split':<8} {'Imgs':>6} {'Normal':>7} {'Hiper':>7} {'Ratio':>7} {'Pacientes':>10}")
print("=" * 58)
for name, d in [("treino", train_df), ("val", val_df), ("teste", test_df)]:
    n    = len(d)
    pos  = (d.label == 1).sum()
    neg  = (d.label == 0).sum()
    pats = d.patient_id.nunique()
    print(f"{name:<8} {n:>6,} {neg:>7,} {pos:>7,} {neg/max(pos,1):>6.1f}:1 {pats:>10,}")
print("=" * 58)


### 4.3 Verificação de Integridade do Split

Antes do treinamento, verificamos três propriedades formais:

1. **Zero leakage** — interseção entre conjuntos de `patient_id` deve ser vazia
2. **Cobertura total** — soma dos splits deve igualar o total do dataset
3. **Estratificação** — ambas as classes presentes nos três splits


In [ ]:
train_pats = set(train_df.patient_id)
val_pats   = set(val_df.patient_id)
test_pats  = set(test_df.patient_id)

print("=== Verificacao de Data Leakage ===")
for label, a, b in [("treino/val", train_pats, val_pats),
                     ("treino/test", train_pats, test_pats),
                     ("val/test", val_pats, test_pats)]:
    overlap = a & b
    print(f"  {label:<12}: {len(overlap)} pacientes em comum  {'[OK]' if not overlap else '[FALHOU]'}")

total_split = len(train_df) + len(val_df) + len(test_df)
print(f"\n=== Cobertura Total ===")
print(f"  Dataset original : {len(df):,}")
print(f"  Soma dos splits  : {total_split:,}")
print(f"  {'[OK] nenhuma imagem perdida' if total_split == len(df) else '[FALHOU]'}")

print(f"\n=== Distribuicao de Classes por Split ===")
for name, d in [("treino", train_df), ("val", val_df), ("teste", test_df)]:
    has_neg = (d.label == 0).sum() > 0
    has_pos = (d.label == 1).sum() > 0
    status = "[OK]" if (has_neg and has_pos) else "[FALHOU]"
    pct_pos = 100 * (d.label == 1).sum() / len(d)
    print(f"  {name:<8}: normal={has_neg}  hipertensiva={has_pos}  pos%={pct_pos:.1f}%  {status}")


---
## 5. Pipeline de Dados

O pipeline conecta o DataFrame anotado ao loop de treinamento via três componentes em
[`src/retino/data/`](https://github.com/Bappoz/fundus-classification/tree/master/src/retino/data):

| Componente | Arquivo | Responsabilidade |
|---|---|---|
| `FundusDataset` | `dataset.py` | Leitura de imagem + aplicação de transforms |
| `make_sampler()` | `dataset.py` | `WeightedRandomSampler` para balanceamento |
| `get_loaders()` | `dataset.py` | DataLoaders de treino/val/teste prontos |

### 5.1 FundusDataset e Augmentação

`FundusDataset` herda de `torch.utils.data.Dataset` e delega as transforms a
`get_transforms(split)` via Albumentations (Buslaev et al., 2020).
Albumentations é preferido ao `torchvision.transforms` pela API NumPy/OpenCV e pela
maior variedade de transformações para imagens médicas.

Ref: [`src/retino/data/dataset.py`](https://github.com/Bappoz/fundus-classification/blob/master/src/retino/data/dataset.py) |
[`src/retino/data/transform.py`](https://github.com/Bappoz/fundus-classification/blob/master/src/retino/data/transform.py)


In [ ]:
from retino.data.dataset import FundusDataset, make_sampler, get_loaders

ds = FundusDataset(train_df, split="train")
img, label = ds[0]

print(f"Shape da imagem : {img.shape}")    # esperado: [3, 224, 224]
print(f"Dtype           : {img.dtype}")    # esperado: torch.float32
print(f"Min / Max       : {img.min():.3f} / {img.max():.3f}")
print(f"Label           : {label.item()}")
print()
print(f"Tamanho treino  : {len(FundusDataset(train_df, split='train')):,}")
print(f"Tamanho val     : {len(FundusDataset(val_df,   split='val')):,}")
print(f"Tamanho teste   : {len(FundusDataset(test_df,  split='test')):,}")


### 5.2 WeightedRandomSampler

Atribui peso `w_i = 1 / freq(classe_i)` a cada amostra. Durante cada epoch, sorteia
`len(dataset)` índices **com reposição** (`replacement=True`) — necessário pois a
classe minoritária precisa ser reutilizada para atingir equilíbrio ~1:1.

Vantagem sobre alternativas:
- vs. **oversampling físico**: sem aumento de custo de I/O
- vs. **undersampling**: nenhum dado da classe majoritária é descartado

Ref: `make_sampler()` — [`src/retino/data/dataset.py`](https://github.com/Bappoz/fundus-classification/blob/master/src/retino/data/dataset.py)


In [ ]:
from collections import Counter

sampler  = make_sampler(train_df)
indices  = list(sampler)
labels   = train_df["label"].values
contagem = Counter(labels[i] for i in indices)

print("Sorteios por classe em 1 epoch (treino):")
print(f"  Normal (0)        : {contagem[0]:,}")
print(f"  Hipertensiva (1)  : {contagem[1]:,}")
print(f"  Ratio amostrado   : {contagem[0]/max(contagem[1],1):.2f}:1  (esperado ~1:1)")
print()
print("Distribuicao real no dataset de treino:")
print(f"  Normal (0)        : {(train_df.label==0).sum():,}")
print(f"  Hipertensiva (1)  : {(train_df.label==1).sum():,}")
print(f"  Ratio real        : {(train_df.label==0).sum()/(train_df.label==1).sum():.1f}:1")


### 5.3 Focal Loss

A Focal Loss (Lin et al., 2017) foi proposta para detecção de objetos com desbalanceamento
extremo, mas é diretamente aplicável a classificação binária.

Formulação:

    FL(pt) = -alpha_t * (1 - pt)^gamma * log(pt)

onde `pt` é a probabilidade prevista para a classe correta, `(1-pt)^gamma` é o **fator
focal** que atenua exemplos fáceis (pt alto, maioria normal) e `alpha_t` é o peso de classe.

Hiperparâmetros:
- `alpha = 0.75` — valor > 0.5 prioriza a classe positiva rara; calibrado empiricamente
- `gamma = 2.0` — valor canônico dos autores; gamma=0 equivale à BCE padrão

Ref: classe `FocalLoss` — [`src/retino/losses.py`](https://github.com/Bappoz/fundus-classification/blob/master/src/retino/losses.py)


In [ ]:
from retino.losses import FocalLoss   # src/retino/losses.py
import torch

loss_fn = FocalLoss(alpha=0.75, gamma=2.0)

# Batch de 8: 7 normais + 1 hipertensiva — ratio 7:1
logits  = torch.randn(8)
targets = torch.tensor([0, 0, 0, 0, 0, 0, 0, 1], dtype=torch.float32)

focal = loss_fn(logits, targets)
bce   = torch.nn.functional.binary_cross_entropy_with_logits(logits, targets)

print(f"Focal Loss : {focal.item():.4f}")
print(f"BCE padrao : {bce.item():.4f}")
print(f"Focal < BCE: {focal.item() < bce.item()}")
print()
print("Focal < BCE confirma que (1-pt)^gamma atenuou os exemplos")
print("faceis (normal com alta confianca), focando nos casos dificeis.")


---
## 6. Testes de Integração

Executa a suíte de testes do repositório
([`tests/`](https://github.com/Bappoz/fundus-classification/tree/master/tests))
sobre uma **amostra pequena** criada localmente, sem precisar do dataset completo.

| Arquivo | O que verifica |
|---|---|
| [`tests/test_split.py`](https://github.com/Bappoz/fundus-classification/blob/master/tests/test_split.py) | Zero leakage, cobertura total, reprodutibilidade, estratificação |
| [`tests/test_loader.py`](https://github.com/Bappoz/fundus-classification/blob/master/tests/test_loader.py) | Parsing de metadados, merge BRSET+ODIR, filter_quality |
| [`tests/test_dataset.py`](https://github.com/Bappoz/fundus-classification/blob/master/tests/test_dataset.py) | Shape/dtype dos tensores, sampler, transforms |

### 6.1 Preparação da Amostra Local

Os testes esperam imagens em `data/sample/` e metadados em `data/meta/`.


In [ ]:
import shutil, pandas as pd
from pathlib import Path

SAMPLE_DIR     = Path("data/sample")
META_DIR_LOCAL = Path("data/meta")

for sub in ["normal", "hr_brset", "hr_odir5k"]:
    (SAMPLE_DIR / sub).mkdir(parents=True, exist_ok=True)
META_DIR_LOCAL.mkdir(parents=True, exist_ok=True)

# 30 imagens por classe — minimo para split 70/15/15 com estratificacao
N = 30
rows = []
for lbl, folder in [(0, "normal"), (1, "hr_brset")]:
    for _, r in df[df.label == lbl].head(N).iterrows():
        dst = SAMPLE_DIR / folder / r["image"]
        if not dst.exists():
            shutil.copy2(r["path"], dst)
        rows.append(r)
for _, r in df[df.source == "odir"].head(10).iterrows():
    dst = SAMPLE_DIR / "hr_odir5k" / r["image"]
    if not dst.exists():
        shutil.copy2(r["path"], dst)
    rows.append(r)

sample_meta = pd.DataFrame(rows)

brset_full = pd.read_excel(cfg.meta_dir / "labels_brset_filt.xlsx")
brset_full.columns = brset_full.columns.str.strip().str.lower()
ids_b = sample_meta[sample_meta.source == "brset"]["patient_id"].str.replace("brset_", "", regex=False)
brset_full[brset_full["patient_id"].astype(str).isin(ids_b)].to_excel(
    META_DIR_LOCAL / "labels_brset_filt.xlsx", index=False
)

odir_full = pd.read_excel(cfg.meta_dir / "data_filt.xlsx")
odir_full.columns = odir_full.columns.str.strip().str.lower()
ids_o = sample_meta[sample_meta.source == "odir"]["patient_id"].str.replace("odir_", "", regex=False)
odir_full[odir_full["patient_id"].astype(str).isin(ids_o)].to_excel(
    META_DIR_LOCAL / "data_filt.xlsx", index=False
)

n_imgs = len(list(SAMPLE_DIR.rglob("*.jpg")))
print(f"Amostra criada: {n_imgs} imagens em {SAMPLE_DIR}/")
print(f"Metadados exportados para {META_DIR_LOCAL}/")


### 6.2 Execução com pytest

> `test_split.py` inclui skip automático quando a amostra é pequena demais
> (`if (df.label==1).sum() < 6: pytest.skip(...)`), garantindo que o CI
> não quebre em datasets mínimos de desenvolvimento.


In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pytest",
     "tests/test_split.py",
     "tests/test_loader.py",
     "tests/test_dataset.py",
     "-v", "--tb=short", "--no-header", "--color=no"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("=== STDERR ===")
    print(result.stderr)


### 6.3 Validação Inline do Split (sem pytest)

Replica as asserções de `test_split.py` diretamente no notebook para inspeção
interativa no Colab sem dependência do pytest.


In [ ]:
from retino.data.loader import build_labels, verify_files, filter_quality
from retino.data.split import split_by_patient
from pathlib import Path

df_s = build_labels(images_dir=Path("data/sample"), meta_dir=Path("data/meta"))
df_s = verify_files(df_s)
df_s = filter_quality(df_s)

tr, va, te = split_by_patient(df_s, seed=42)

sets = [set(d.patient_id) for d in (tr, va, te)]
assert sets[0].isdisjoint(sets[1]) and sets[0].isdisjoint(sets[2]) and sets[1].isdisjoint(sets[2])
print("[OK] test_no_leakage")

assert len(tr) + len(va) + len(te) == len(df_s)
print(f"[OK] test_covers_all_data  ({len(df_s)} imagens)")

n = len(df_s)
assert abs(len(tr)/n - 0.70) < 0.05
assert abs(len(va)/n - 0.15) < 0.05
assert abs(len(te)/n - 0.15) < 0.05
print(f"[OK] test_ratios_approximate  treino={len(tr)/n:.0%}  val={len(va)/n:.0%}  teste={len(te)/n:.0%}")

tr2, va2, _ = split_by_patient(df_s, seed=42)
assert set(tr.patient_id) == set(tr2.patient_id)
print("[OK] test_reproducible")

print("\nTodos os testes passaram.")
